In [6]:

# 2024-06-12: .env 파일에서 키움 API 설정값 불러오기 (테스트/예제용)
import os
from pathlib import Path
from dotenv import load_dotenv
from pykiwoom_rest import KiwoomRest
# Jupyter 환경에서는 __file__ 미정의 → 현재 워킹 디렉토리 기준으로 상위 폴더의 .env 지정
env_path = (Path.cwd().parent / ".env").resolve()
load_dotenv(dotenv_path=env_path)

ACCOUNT_NO = os.getenv("ACCOUNT_NO")
KIWOOM_APPKEY = os.getenv("KIWOOM_APPKEY")
KIWOOM_APPSECRET = os.getenv("KIWOOM_APPSECRET")


agent = KiwoomRest(
    account_no=ACCOUNT_NO,
    appkey=KIWOOM_APPKEY,
    appsecret=KIWOOM_APPSECRET,
    env_path=env_path,
    use_mock=False,
)

if agent:
    print("API initialized successfully")


API initialized successfully


In [ ]:
program = agent.get_hourly_program_trading_paginated("005930", "20250922", "1")['output']

# 순매수금액(prm_netprps_amt), 순매수금액증감(prm_netprps_amt_irds) 기반 추세 분석
def parse_amount(val):
    # +, - 기호 및 쉼표 제거 후 int 변환
    if isinstance(val, str):
        val = val.replace("+", "").replace(",", "")
        try:
            return int(val)
        except Exception:
            return 0
    return 0

# 순매수금액 리스트
net_amounts = [parse_amount(tick.get('prm_netprps_amt', '0')) for tick in program]
# 순매수금액증감 리스트
net_amount_changes = [parse_amount(tick.get('prm_netprps_amt_irds', '0')) for tick in program]

# 추세 판단: 단순히 첫 값과 마지막 값 비교 (증가/감소/유지)
if not net_amounts or len(net_amounts) < 2:
    trend = "데이터 부족"
else:
    if net_amounts[-1] > net_amounts[0]:
        trend = "순매수금액 증가"
    elif net_amounts[-1] < net_amounts[0]:
        trend = "순매수금액 감소"
    else:
        trend = "순매수금액 변화 없음"

# 결과 출력
{
    "시작시각": program[0]['tm'] if program else None,
    "종료시각": program[-1]['tm'] if program else None,
    "시작_순매수금액": net_amounts[0] if net_amounts else None,
    "종료_순매수금액": net_amounts[-1] if net_amounts else None,
    "총_증감": net_amounts[-1] - net_amounts[0] if len(net_amounts) >= 2 else None,
    "추세": trend,
    "순매수금액증감_시퀀스": net_amount_changes[:5],  # 앞 5개만 예시로
}

[DEBUG] tick[0]: {'tm': '134657', 'cur_prc': '+82900', 'pre_sig': '2', 'pred_pre': '+3200', 'flu_rt': '+4.02', 'trde_qty': '22031682', 'prm_sell_amt': '154516', 'prm_buy_amt': '721089', 'prm_netprps_amt': '+566573', 'prm_netprps_amt_irds': '+406', 'prm_sell_qty': '1857069', 'prm_buy_qty': '8681629', 'prm_netprps_qty': '+6824560', 'prm_netprps_qty_irds': '+4902', 'base_pric_tm': '', 'dbrt_trde_rpy_sum': '', 'remn_rcvord_sum': '', 'stex_tp': 'KRX'}
[DEBUG] tick[1]: {'tm': '134651', 'cur_prc': '+82900', 'pre_sig': '2', 'pred_pre': '+3200', 'flu_rt': '+4.02', 'trde_qty': '22024115', 'prm_sell_amt': '154506', 'prm_buy_amt': '720673', 'prm_netprps_amt': '+566167', 'prm_netprps_amt_irds': '--49', 'prm_sell_qty': '1856941', 'prm_buy_qty': '8676599', 'prm_netprps_qty': '+6819658', 'prm_netprps_qty_irds': '--590', 'base_pric_tm': '', 'dbrt_trde_rpy_sum': '', 'remn_rcvord_sum': '', 'stex_tp': 'KRX'}
[DEBUG] 필드명 샘플: {'tm', 'prm_buy_amt', 'prm_netprps_qty_irds', 'pre_sig', 'remn_rcvord_sum', 'prm_s

{'시작시각': '085004',
 '종료시각': '134657',
 '시작_순매수금액': 0,
 '종료_순매수금액': 566573,
 '총_증감': 566573,
 '추세': '순매수금액 증가',
 '순매수금액_시퀀스': [0, 0, 0, 0, 0],
 '순매수금액증감_시퀀스': [0, 0, 0, 0, 0],
 '원본_샘플': [{'tm': '085004',
   'cur_prc': '79700',
   'pre_sig': '3',
   'pred_pre': '0',
   'flu_rt': '0.00',
   'trde_qty': '151',
   'prm_sell_amt': '0',
   'prm_buy_amt': '0',
   'prm_netprps_amt': '0',
   'prm_netprps_amt_irds': '0',
   'prm_sell_qty': '0',
   'prm_buy_qty': '0',
   'prm_netprps_qty': '0',
   'prm_netprps_qty_irds': '0',
   'base_pric_tm': '',
   'dbrt_trde_rpy_sum': '',
   'remn_rcvord_sum': '',
   'stex_tp': 'KRX'},
  {'tm': '085014',
   'cur_prc': '79700',
   'pre_sig': '3',
   'pred_pre': '0',
   'flu_rt': '0.00',
   'trde_qty': '151',
   'prm_sell_amt': '0',
   'prm_buy_amt': '0',
   'prm_netprps_amt': '0',
   'prm_netprps_amt_irds': '0',
   'prm_sell_qty': '0',
   'prm_buy_qty': '0',
   'prm_netprps_qty': '0',
   'prm_netprps_qty_irds': '0',
   'base_pric_tm': '',
   'dbrt_trde_rpy_sum'

## 프로그램매매 추세 분석 (고도화 버전)

장 시작부터 현재까지의 프로그램매매 데이터를 수집하여 스마트하게 추세를 분석합니다.
- 전반부/후반부 평균 비교
- 선형 회귀 기울기 분석
- 변동성 및 상승 비율 계산
- 종합적인 추세 판단 (상승/하락/횡보)

In [ ]:
import numpy as np
from datetime import datetime

class ProgramTradingTrendAnalyzer:
    """프로그램매매 추세 분석 클래스"""
    
    def __init__(self, agent):
        self.agent = agent
    
    def parse_amount(self, val):
        """금액 문자열을 정수로 변환 (음수 처리 포함)"""
        if isinstance(val, str):
            # -- 기호는 음수를 의미
            is_negative = val.startswith('--')
            # 모든 기호 제거
            val_clean = val.replace("+", "").replace(",", "").replace("-", "")
            try:
                amt = int(val_clean)
                return -amt if is_negative else amt
            except:
                return 0
        return 0
    
    def fetch_all_day_data(self, stock_code, date=None):
        """장 시작부터 현재까지 모든 데이터 수집"""
        if date is None:
            date = datetime.now().strftime("%Y%m%d")
        
        print(f"전체 데이터 조회 중... [{stock_code}] {date}")
        result = self.agent.get_hourly_program_trading_paginated(stock_code, date, "1")
        
        if 'output' in result and result['output']:
            # 시간순 정렬
            data = sorted(result['output'], key=lambda x: x.get('tm', '000000'))
            # 장 시작(09:00) 이후 데이터만 필터링
            filtered_data = [d for d in data if d.get('tm', '000000') >= '090000']
            print(f"  → {len(filtered_data)}개 시점 데이터 수집 완료")
            return filtered_data
        return []
    
    def analyze_trend_smart(self, data):
        """스마트한 추세 분석"""
        if not data or len(data) < 2:
            return {"trend": "데이터 부족", "details": {}}
        
        # 순매수금액 추출
        net_amounts = [self.parse_amount(d.get('prm_netprps_amt', '0')) for d in data]
        net_quantities = [self.parse_amount(d.get('prm_netprps_qty', '0')) for d in data]
        
        # 1. 전반부/후반부 평균 비교
        mid_point = len(net_amounts) // 2
        first_half_avg = np.mean(net_amounts[:mid_point]) if mid_point > 0 else 0
        second_half_avg = np.mean(net_amounts[mid_point:]) if mid_point < len(net_amounts) else 0
        
        # 2. 선형 회귀를 통한 기울기 계산
        x = np.arange(len(net_amounts))
        coefficients = np.polyfit(x, net_amounts, 1)
        slope = coefficients[0]
        
        # 3. 변동성 계산
        volatility = np.std(net_amounts) if len(net_amounts) > 1 else 0
        avg_amount = np.mean(net_amounts)
        cv = (volatility / avg_amount * 100) if avg_amount != 0 else 0
        
        # 4. 구간별 증감 패턴
        changes = [net_amounts[i+1] - net_amounts[i] for i in range(len(net_amounts)-1)]
        positive_changes = sum(1 for c in changes if c > 0)
        positive_ratio = positive_changes / len(changes) if changes else 0
        
        # 5. 추세 판단 (종합)
        if abs(cv) < 5:  # 변동성이 매우 낮음
            trend = "횡보"
            confidence = "높음"
        elif slope > 0 and second_half_avg > first_half_avg * 1.05:
            trend = "강한 상승"
            confidence = "높음" if positive_ratio > 0.6 else "중간"
        elif slope > 0 and second_half_avg > first_half_avg:
            trend = "상승"
            confidence = "중간"
        elif slope < 0 and second_half_avg < first_half_avg * 0.95:
            trend = "강한 하락"
            confidence = "높음" if positive_ratio < 0.4 else "중간"
        elif slope < 0 and second_half_avg < first_half_avg:
            trend = "하락"
            confidence = "중간"
        else:
            trend = "횡보"
            confidence = "낮음"
        
        return {
            "trend": trend,
            "confidence": confidence,
            "details": {
                "first_half_avg": first_half_avg,
                "second_half_avg": second_half_avg,
                "change_ratio": ((second_half_avg - first_half_avg) / first_half_avg * 100) if first_half_avg != 0 else 0,
                "slope": slope,
                "volatility_cv": cv,
                "positive_change_ratio": positive_ratio * 100,
                "total_change": net_amounts[-1] - net_amounts[0],
                "start_value": net_amounts[0],
                "end_value": net_amounts[-1],
                "max_value": max(net_amounts),
                "min_value": min(net_amounts)
            },
            "quantities": {
                "start": net_quantities[0] if net_quantities else 0,
                "end": net_quantities[-1] if net_quantities else 0,
                "change": (net_quantities[-1] - net_quantities[0]) if net_quantities and len(net_quantities) > 1 else 0
            },
            "time_range": {
                "start": data[0].get('tm', 'N/A') if data else 'N/A',
                "end": data[-1].get('tm', 'N/A') if data else 'N/A'
            }
        }

# 분석기 초기화
analyzer = ProgramTradingTrendAnalyzer(agent)
print("프로그램매매 추세 분석기 초기화 완료")

### 삼성전자 분석

In [ ]:
# 삼성전자 프로그램매매 추세 분석
stock_code = "005930"
stock_name = "삼성전자"

print(f"{'='*70}")
print(f"📊 {stock_name}({stock_code}) 프로그램매매 추세 분석")
print(f"{'='*70}")

# 데이터 수집
data = analyzer.fetch_all_day_data(stock_code)

if data:
    # 추세 분석
    analysis = analyzer.analyze_trend_smart(data)
    
    print(f"\n⏰ 시간 범위: {analysis['time_range']['start']} ~ {analysis['time_range']['end']}")
    
    print(f"\n💰 순매수금액 분석:")
    print(f"  • 시작: {analysis['details']['start_value']:,}백만원")
    print(f"  • 현재: {analysis['details']['end_value']:,}백만원")
    print(f"  • 변화: {analysis['details']['total_change']:+,}백만원")
    print(f"  • 최대: {analysis['details']['max_value']:,}백만원")
    print(f"  • 최소: {analysis['details']['min_value']:,}백만원")
    
    print(f"\n📊 구간 분석:")
    print(f"  • 전반부 평균: {analysis['details']['first_half_avg']:,.0f}백만원")
    print(f"  • 후반부 평균: {analysis['details']['second_half_avg']:,.0f}백만원")
    print(f"  • 변화율: {analysis['details']['change_ratio']:+.2f}%")
    
    print(f"\n📉 추세 지표:")
    print(f"  • 선형 기울기: {analysis['details']['slope']:+.2f}")
    print(f"  • 변동성(CV): {analysis['details']['volatility_cv']:.2f}%")
    print(f"  • 상승 비율: {analysis['details']['positive_change_ratio']:.1f}%")
    
    print(f"\n🎯 추세 판단: **{analysis['trend']}** (신뢰도: {analysis['confidence']})")
    
    print(f"\n📦 순매수수량:")
    print(f"  • 시작: {analysis['quantities']['start']:,}주")
    print(f"  • 현재: {analysis['quantities']['end']:,}주")
    print(f"  • 변화: {analysis['quantities']['change']:+,}주")
else:
    print("데이터를 수집할 수 없습니다.")

### SK하이닉스 분석

In [ ]:
# SK하이닉스 프로그램매매 추세 분석
stock_code = "000660"
stock_name = "SK하이닉스"

print(f"{'='*70}")
print(f"📊 {stock_name}({stock_code}) 프로그램매매 추세 분석")
print(f"{'='*70}")

# 데이터 수집
data = analyzer.fetch_all_day_data(stock_code)

if data:
    # 추세 분석
    analysis = analyzer.analyze_trend_smart(data)
    
    print(f"\n⏰ 시간 범위: {analysis['time_range']['start']} ~ {analysis['time_range']['end']}")
    
    print(f"\n💰 순매수금액 분석:")
    print(f"  • 시작: {analysis['details']['start_value']:,}백만원")
    print(f"  • 현재: {analysis['details']['end_value']:,}백만원")
    print(f"  • 변화: {analysis['details']['total_change']:+,}백만원")
    
    # 음수 처리를 위한 표시
    if analysis['details']['end_value'] < 0:
        print(f"  ⚠️ 현재 순매도 상태입니다!")
    
    print(f"  • 최대: {analysis['details']['max_value']:,}백만원")
    print(f"  • 최소: {analysis['details']['min_value']:,}백만원")
    
    print(f"\n📊 구간 분석:")
    print(f"  • 전반부 평균: {analysis['details']['first_half_avg']:,.0f}백만원")
    print(f"  • 후반부 평균: {analysis['details']['second_half_avg']:,.0f}백만원")
    print(f"  • 변화율: {analysis['details']['change_ratio']:+.2f}%")
    
    print(f"\n📉 추세 지표:")
    print(f"  • 선형 기울기: {analysis['details']['slope']:+.2f}")
    print(f"  • 변동성(CV): {abs(analysis['details']['volatility_cv']):.2f}%")
    print(f"  • 상승 비율: {analysis['details']['positive_change_ratio']:.1f}%")
    
    print(f"\n🎯 추세 판단: **{analysis['trend']}** (신뢰도: {analysis['confidence']})")
    
    print(f"\n📦 순매수수량:")
    print(f"  • 시작: {analysis['quantities']['start']:,}주")
    print(f"  • 현재: {analysis['quantities']['end']:,}주")
    print(f"  • 변화: {analysis['quantities']['change']:+,}주")
else:
    print("데이터를 수집할 수 없습니다.")

### 여러 종목 동시 비교

In [ ]:
# 여러 종목 한번에 분석
stocks = [
    ("005930", "삼성전자"),
    ("000660", "SK하이닉스"),
    ("005380", "현대차"),
    ("051910", "LG화학")
]

results = []

for stock_code, stock_name in stocks:
    print(f"\n분석 중: {stock_name}({stock_code})")
    
    # 데이터 수집
    data = analyzer.fetch_all_day_data(stock_code)
    
    if data:
        # 추세 분석
        analysis = analyzer.analyze_trend_smart(data)
        
        results.append({
            "종목명": stock_name,
            "종목코드": stock_code,
            "추세": analysis['trend'],
            "신뢰도": analysis['confidence'],
            "순매수금액": analysis['details']['end_value'],
            "변화": analysis['details']['total_change'],
            "변화율": analysis['details']['change_ratio'],
            "상승비율": analysis['details']['positive_change_ratio']
        })

# 결과 요약 테이블
print(f"\n{'='*100}")
print(f"📊 종합 분석 결과")
print(f"{'='*100}")
print(f"{'종목명':^10} {'종목코드':^8} {'추세':^12} {'신뢰도':^6} {'순매수금액':>12} {'변화':>12} {'변화율':>8} {'상승비율':>8}")
print(f"{'-'*100}")

for r in results:
    print(f"{r['종목명']:10} {r['종목코드']:8} {r['추세']:12} {r['신뢰도']:6} "
          f"{r['순매수금액']:12,} {r['변화']:+12,} {r['변화율']:+7.1f}% {r['상승비율']:7.1f}%")

# 추세별 그룹핑
print(f"\n📈 추세별 분류:")
for trend_type in ["강한 상승", "상승", "횡보", "하락", "강한 하락"]:
    stocks_in_trend = [r['종목명'] for r in results if r['추세'] == trend_type]
    if stocks_in_trend:
        print(f"  • {trend_type}: {', '.join(stocks_in_trend)}")